# Cloud-Native Sentinel-2 Workflow: Guwahati, Assam

**STAC · Planetary Computer · odc-stac · Xarray · Zarr · NDVI**

This notebook shows an end-to-end, cloud-native Earth observation workflow over **Guwahati, Assam (India)**. Instead of downloading whole satellite scenes, it queries a STAC catalogue, builds a *lazy* data cube from Cloud Optimised GeoTIFFs (COGs), and only pulls the pixels it needs into a local, chunked **Zarr** store for fast re-use.

| Step | What happens | Pixels downloaded |
|---|---|---|
| 1 | Connect to the Microsoft Planetary Computer STAC API | 0 |
| 2 | Search Sentinel-2 L2A scenes (AOI, dates, cloud cover) | 0 |
| 3 | Build a lazy 3D Xarray dataset with `odc-stac` | 0 |
| 4 | Persist the cube to a local Zarr store | Yes (once) |
| 5 | Re-open from Zarr and derive a False-Colour Composite and NDVI | No (local) |

**Data:** Sentinel-2 Level-2A (`sentinel-2-l2a`), bands B08 (NIR), B04 (Red), B03 (Green)
**Area of interest:** `91.669, 26.097, 91.872, 26.218` (lon/lat bounding box, EPSG:4326)
**Period:** 1 Oct – 30 Nov 2026, scenes with < 15 % cloud cover

## 0. Setup

In [1]:
import warnings

import numpy as np
import xarray as xr
import zarr
import pystac_client
import planetary_computer
import odc.stac
import plotly.express as px

# rasterio emits a harmless georeferencing warning when plotly wraps arrays
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="Dataset has no geotransform")

## 1. Connect to the STAC API

The Planetary Computer hosts assets in Azure Blob Storage behind short-lived SAS tokens. Passing `sign_inplace` as a *modifier* signs every asset URL automatically.

In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

## 2. Search for scenes

Only **metadata** is queried here. Filters: collection, bounding box (drawn with [bboxfinder.com](http://bboxfinder.com)), date range (`YYYY-MM-DD/YYYY-MM-DD`) and a cloud-cover threshold.

In [3]:
AOI_BBOX = (91.669235, 26.097488, 91.871796, 26.217671)  # Guwahati, Assam
DATE_RANGE = "2026-10-01/2026-11-30"

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=AOI_BBOX,
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": 15}},
)

In [4]:
items = search.item_collection()
print(f"Total Scenes Found: {len(items)}")

Total Scenes Found: 13


A quick look at what was found, sorted by acquisition date:

In [5]:
for item in sorted(items, key=lambda i: i.datetime):
    print(f"{item.datetime:%Y-%m-%d}  |  cloud cover: {item.properties['eo:cloud_cover']:5.1f} %  |  {item.id}")

2026-10-07  |  cloud cover:   5.3 %  |  S2B_MSIL2A_20261007T042659_R133_T46RCQ_20261007T080800
2026-10-12  |  cloud cover:   0.2 %  |  S2C_MSIL2A_20261012T042721_R133_T46RCQ_20261012T080303
2026-10-12  |  cloud cover:   0.1 %  |  S2C_MSIL2A_20261012T042721_R133_T46RCQ_20261012T070609
2026-10-12  |  cloud cover:   0.1 %  |  S2C_MSIL2A_20261012T042721_R133_T46RCP_20261012T080303
2026-10-12  |  cloud cover:   0.0 %  |  S2C_MSIL2A_20261012T042721_R133_T46RCP_20261012T070609
2026-10-17  |  cloud cover:  14.0 %  |  S2B_MSIL2A_20261017T042749_R133_T46RCQ_20261017T081241
2026-10-22  |  cloud cover:   0.0 %  |  S2C_MSIL2A_20261022T042821_R133_T46RCQ_20261022T075358
2026-10-22  |  cloud cover:   2.1 %  |  S2C_MSIL2A_20261022T042821_R133_T46RCP_20261022T075358
2026-10-27  |  cloud cover:   9.3 %  |  S2B_MSIL2A_20261027T042859_R133_T46RCQ_20261027T121537
2026-10-27  |  cloud cover:   9.7 %  |  S2B_MSIL2A_20261027T042859_R133_T46RCP_20261027T121537
2026-11-06  |  cloud cover:   3.1 %  |  S2B_MSIL2A

## 3. Build a lazy data cube

`odc.stac.load` converts the STAC item collection into an `xarray.Dataset` with dimensions `(time, y, x)`. Nothing is downloaded at this stage; Dask only stores the *recipe* for reading each chunk.

- `resolution=200` downsamples to 200 m for a fast, lightweight demo. Use `10` for full Sentinel-2 resolution.
- `groupby="solarday"` merges tiles captured on the same solar day into one time step.
- `crs="EPSG:32646"` reprojects to UTM zone 46N, which suits Assam.

In [6]:
cog_ds = odc.stac.load(
    items,
    bands=["B08", "B04", "B03"],
    resolution=200,
    groupby="solarday",
    chunks={"x": 512, "y": 512},
    crs="EPSG:32646",
)
cog_ds

<xarray.Dataset> Size: 7MB
Dimensions:      (y: 1049, x: 549, time: 1)
Coordinates:
  * y            (y) float64 8kB 3e+06 3e+06 3e+06 ... 2.79e+06 2.79e+06
  * x            (x) float64 4kB 3.001e+05 3.003e+05 ... 4.095e+05 4.097e+05
  * time         (time) datetime64[us] 8B 2026-10-07T04:26:59.024000
    spatial_ref  int32 4B 32646
Data variables:
    B08          (time, y, x) float32 2MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    B04          (time, y, x) float32 2MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    B03          (time, y, x) float32 2MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>

### Quick look: False-Colour Composite (NIR-Red-Green)

Mapping NIR to the red channel makes healthy vegetation appear bright red, water dark and built-up areas grey-cyan. Computing this plot triggers the first real data read.

In [7]:
fcc_preview = (
    cog_ds[["B08", "B04", "B03"]]
    .isel(time=0)
    .to_array("band")
    .transpose("y", "x", "band")
)
px.imshow(fcc_preview, zmin=0, zmax=3000).show()

## 4. Persist to Zarr

Writing to Zarr downloads the pixels once and stores them as compressed, chunked arrays on local disk. Later analysis then reads from disk instead of the network.

In [8]:
cog_ds.to_zarr("guwahati_s2.zarr", mode="w")

## 5. Re-open the local store

The Zarr store opens almost instantly because only metadata is read; chunks are loaded on demand.

In [9]:
ds = xr.open_zarr("guwahati_s2.zarr")
ds

<xarray.Dataset> Size: 7MB
Dimensions:      (time: 1, y: 1049, x: 549)
Coordinates:
  * time         (time) datetime64[ns] 8B 2026-10-07T04:26:59.024000
  * y            (y) float64 8kB 3e+06 3e+06 3e+06 ... 2.79e+06 2.79e+06
  * x            (x) float64 4kB 3.001e+05 3.003e+05 ... 4.095e+05 4.097e+05
Data variables:
    B03          (time, y, x) float32 2MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    B04          (time, y, x) float32 2MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    B08          (time, y, x) float32 2MB dask.array<chunksize=(1, 512, 512), meta=np.ndarray>
    spatial_ref  int32 4B ...

## 6. False-Colour Composite (north-up)

In [10]:
fcc = (
    ds[["B08", "B04", "B03"]]
    .isel(time=0)
    .to_array("band")
    .transpose("y", "x", "band")
)

fig = px.imshow(
    fcc,
    zmin=0,
    zmax=3000,
    origin="upper",
    width=800,
    height=800,
    title=f"False-Colour Composite (NIR-R-G), {str(ds.time.values[0])[:10]}",
)
fig.show()

## 7. NDVI

$$NDVI = \frac{NIR - Red}{NIR + Red}$$

Sentinel-2 L2A scenes processed with baseline ≥ 04.00 (since January 2022) carry a **+1000 DN radiometric offset**. The helper below masks no-data (`0`) and converts digital numbers to surface reflectance before computing the index. If your cube is already float reflectance, the step is skipped.

In [11]:
def to_reflectance(da: xr.DataArray) -> xr.DataArray:
    """Mask no-data and convert Sentinel-2 L2A DNs to reflectance (0-1)."""
    if np.issubdtype(da.dtype, np.floating):
        return da
    return da.astype("float32").where(da != 0).pipe(lambda x: (x - 1000) / 10000)


nir = to_reflectance(ds["B08"])
red = to_reflectance(ds["B04"])

ndvi = ((nir - red) / (nir + red)).clip(-1, 1).rename("ndvi")
ndvi_median = ndvi.median("time", skipna=True)  # cloud-robust temporal composite

In [12]:
px.imshow(
    ndvi.isel(time=0),
    color_continuous_scale="RdYlGn",
    zmin=-0.2,
    zmax=0.8,
    origin="upper",
    width=850,
    height=800,
    title=f"NDVI, {str(ds.time.values[0])[:10]}",
).show()

### Temporal median NDVI

Taking the median across all clear scenes suppresses residual cloud, shadow and haze, giving a more stable picture of vegetation across the Oct–Nov window.

In [13]:
px.imshow(
    ndvi_median,
    color_continuous_scale="RdYlGn",
    zmin=-0.2,
    zmax=0.8,
    origin="upper",
    width=850,
    height=800,
    title="Median NDVI, Oct-Nov 2026",
).show()

## Summary

- A STAC search over Sentinel-2 L2A returned **13 low-cloud scenes** for Guwahati with no pixels downloaded.
- `odc-stac` assembled them into a lazy, Dask-backed `(time, y, x)` cube.
- The cube was written once to **Zarr** and re-opened locally for fast analysis.
- A false-colour composite and **NDVI** (single date and temporal median) were produced from the local store.

**Possible extensions:** run at 10 m resolution, add the SCL band for cloud masking, build an NDVI time series, or compare against the monsoon season.

*Data: Copernicus Sentinel-2 (ESA), accessed via Microsoft Planetary Computer.*